# 投票法：最简单的集成策略
> 难度标签：中级 | 预计时长：15-25分钟 | 前置知识：无学习经验


> 所属模块：07_集成学习 | 源文件：投票法_Voting.py | 硬投票/软投票、权重调节

## 概述

投票法把多个不同模型的预测结果通过投票（分类）或平均（回归）合并。硬投票取多数类，软投票取概率平均。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
from sklearn.datasets import load_iris, make_regression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import VotingClassifier, VotingRegressor
from sklearn.linear_model import LogisticRegression
# --- 导入库 ---
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

## 数学原理

### 1. 投票法的基本框架

投票法集成通过聚合多个独立训练的模型的预测来提升性能。设 $K$ 个模型为 $f_1, f_2, \ldots, f_K$，分类问题有 $C$ 个类别。

### 2. 硬投票（Hard Voting / Majority Voting）

最终预测为多数表决：

$$\hat{y} = \arg\max_{c \in \{1,\ldots,C\}} \sum_{k=1}^{K} \mathbb{I}[f_k(x) = c]$$

即选择被最多模型预测的类别。每个模型权重相等，票数相同。

### 3. 软投票（Soft Voting / Weighted Average）

当模型能输出类别概率 $p_k(c|x)$ 时，聚合概率：

$$\hat{y} = \arg\max_c \sum_{k=1}^{K} w_k \cdot p_k(c|x)$$

其中 $w_k$ 是第 $k$ 个模型的权重。代码中 `voting="soft"` 需要模型支持 `predict_proba`。

软投票的优势：考虑了模型的**置信度**。一个模型以 0.99 概率预测类别 A，比以 0.51 概率预测类别 A 的信息量更大。

### 4. 加权投票

引入权重 $w_k$ 反映各模型的可靠程度：

$$\hat{y} = \arg\max_c \sum_{k=1}^{K} w_k \cdot p_k(c|x), \quad w_k \geq 0$$

代码中尝试了不同权重组合：
- `weights=[1,1,1,1,1]`：等权，等价于软投票
- `weights=[2,1,2,2,1]`：对 RF、GB、SVM 给予更高信任
- `weights=[3,1,3,3,1]`：进一步放大强模型的权重

### 5. 投票法有效的条件

投票法提升性能的数学条件（Condorcet 陪审团定理的推广）：

设每个模型独立分类正确的概率为 $p > 0.5$，$K$ 个模型多数投票正确的概率：

$$P(\text{correct}) = \sum_{k=\lceil K/2 \rceil}^{K} \binom{K}{k} p^k (1-p)^{K-k}$$

当 $p > 0.5$ 且 $K \to \infty$ 时，$P(\text{correct}) \to 1$。

关键前提：模型的错误**不相关**。模型多样性越高，投票法效果越好。

### 6. 硬投票 vs 软投票的数学差异

| 方法 | 聚合方式 | 信息利用 | 适用条件 |
|------|----------|----------|----------|
| 硬投票 | 多数表决 $\arg\max \sum \mathbb{I}$ | 仅用类别标签 | 任何分类器 |
| 软投票 | 概率加权平均 $\arg\max \sum w_k p_k$ | 用概率置信度 | 需要 `predict_proba` |

软投票在概率校准良好时通常优于硬投票。

### 7. 代码与数学的对应

| 代码 | 数学含义 |
|------|----------|
| `voting="hard"` | $\hat{y} = \arg\max_c \sum \mathbb{I}[f_k(x)=c]$ |
| `voting="soft"` | $\hat{y} = \arg\max_c \sum w_k p_k(c|x)$ |
| `weights=[2,1,2,2,1]` | 加权软投票，$w = (2,1,2,2,1)$ |
| `predict_proba=True` | SVC 输出概率用于软投票 |
| `cross_val_score` | 单模型的 5 折交叉验证准确率 |
| 5 个不同的模型 | $f_1, \ldots, f_5$，多样性是投票法有效的关键 |

### 1. 加载数据

首先加载数据集，为后续训练和评估做准备。

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

### 2. 定义各模型

运行 2. 定义各模型 的代码，观察算法在该环节的行为。

In [ ]:
models = [
    ("lr", LogisticRegression(max_iter=200, random_state=42)),
    ("dt", DecisionTreeClassifier(max_depth=5, random_state=42)),
    ("rf", RandomForestClassifier(n_estimators=50, random_state=42)),
    ("svm", SVC(kernel="rbf", probability=True, random_state=42)),
# --- 继续 ---
    ("knn", KNeighborsClassifier(n_neighbors=5)),
]

### 3. 各模型单独表现

运行 3. 各模型单独表现 的代码，观察算法在该环节的行为。

In [ ]:
print("=== 各模型单独表现 ===")
for name, model in models:
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring="accuracy")
    model.fit(X_train, y_train)
    test_acc = model.score(X_test, y_test)
# --- 输出结果 ---
    print(f"  {name:>5}: CV={scores.mean():.4f}±{scores.std():.4f}, Test={test_acc:.4f}")

### 4. 硬投票（多数表决）

运行 4. 硬投票（多数表决） 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 硬投票 (hard voting) ===")
voting_hard = VotingClassifier(estimators=models, voting="hard", n_jobs=-1)
voting_hard.fit(X_train, y_train)
print(f"测试准确率: {voting_hard.score(X_test, y_test):.4f}")

### 5. 软投票（概率平均）

运行 5. 软投票（概率平均） 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 软投票 (soft voting) ===")
voting_soft = VotingClassifier(estimators=models, voting="soft", n_jobs=-1)
voting_soft.fit(X_train, y_train)
print(f"测试准确率: {voting_soft.score(X_test, y_test):.4f}")

### 6. 加权投票

运行 6. 加权投票 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 加权投票 ===")
# 给不同模型不同权重
for weights in [[1,1,1,1,1], [2,1,2,2,1], [3,1,3,3,1]]:
    vw = VotingClassifier(estimators=models, voting="soft", weights=weights, n_jobs=-1)
    vw.fit(X_train, y_train)
    print(f"  weights={weights}: 测试准确率={vw.score(X_test, y_test):.4f}")

### 7. 预测概率

使用训练好的模型进行预测，观察输出结果。

In [ ]:
print("\n=== 软投票预测概率（前 5 个样本）===")
voting_soft.fit(X_train, y_train)
y_prob = voting_soft.predict_proba(X_test[:5])
y_pred = voting_soft.predict(X_test[:5])
for i in range(5):
# --- 循环处理 ---
    probs = ", ".join(f"{p:.3f}" for p in y_prob[i])
    print(f"  样本{i+1}: 预测={y_pred[i]}, 概率=[{probs}], 真实={y_test[i]}")

### 8. 投票法回归

在回归任务上拟合模型，观察预测值与真实值的偏差。

In [ ]:
print("\n=== Voting 回归 ===")
X_r, y_r = make_regression(n_samples=300, n_features=10, noise=10, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_r, y_r, test_size=0.2, random_state=42)

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge

reg_models = [
    ("dt", DecisionTreeRegressor(max_depth=5, random_state=42)),
    ("rf", RandomForestRegressor(n_estimators=50, random_state=42)),
    ("gb", GradientBoostingRegressor(n_estimators=50, random_state=42)),
    ("ridge", Ridge()),
# --- 继续 ---
]
voting_reg = VotingRegressor(estimators=reg_models)
voting_reg.fit(X_tr, y_tr)
print(f"Voting 回归 R²: {voting_reg.score(X_te, y_te):.4f}")
for name, model in reg_models:
# --- 训练模型 ---
    model.fit(X_tr, y_tr)
    print(f"  {name:>5} R²: {model.score(X_te, y_te):.4f}")

print("\n=== 投票法要点 ===")
print("- 硬投票: 多数类别获胜（不需要概率输出）")
print("- 软投票: 概率平均后选最大（需要模型支持 predict_proba）")
print("- 加权投票: 给更好的模型更高权重")
print("- 效果好当：模型多样性强且各模型准确率 > 随机")
# --- 输出结果 ---
print("- 简单有效，常作为 baseline 或最终集成手段")

## 关键代码解释

```python
from sklearn.ensemble import VotingClassifier
voting = VotingClassifier(
    estimators=[("lr", lr), ("rf", rf), ("svm", svm)],
    voting="soft", weights=[1, 2, 1]
)
```

## 注意事项

1. 软投票通常优于硬投票（利用了概率信息）
2. 模型越多样化，集成效果越好
3. 权重可以基于验证集性能手动调节

## 总结与延伸

以上代码展示了 **投票法 Voting** 的完整流程。

**解读要点**：
- 对比 **单模型 vs 集成模型** 的性能提升
- 关注 **特征重要性** 排名
- 观察 OOB 分数（随机森林）或 early stopping 轮次（XGBoost）

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **加权平均 vs 简单平均**：调权重可能过拟合
- **Blending**：用验证集预测值训练元模型
- **Stacking**：更系统化的多层集成
